In [44]:
import os
import copy
import json
import time
import random
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [45]:


SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_ROOT = Path("./prepared_balanced_nilm")
CHECKPOINT_DIR = Path("./checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

APPLIANCE = "washing_machine"
TARGET_BUILDING = "building_04"

TRAIN_BUILDINGS = ["building_01", "building_02"]
VAL_BUILDING = "building_03"

WINDOW_SIZE = 129
BATCH_SIZE = 256
EVAL_BATCH_SIZE = 2048
NUM_WORKERS = 0

CLASSIFIER_EPOCHS = 6
REGRESSOR_EPOCHS = 6
CALIBRATION_EPOCHS = 3

CLASSIFIER_LR = 1e-4
REGRESSOR_LR = 5e-5

ACTIVITY_THRESHOLD = 2.0
CALIBRATION_RATIO = 0.05
CALIBRATION_MODE = "all"   
PROB_THRESHOLD = 0.5
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [46]:
def building_dir(building):
    return DATA_ROOT / building

def csv_path(building):
    return building_dir(building) / "prepared_timeseries.csv"

def metadata_path(building):
    return building_dir(building) / "metadata.json"

def center_path(building, appliance, kind):
    return building_dir(building) / f"{appliance}_{kind}_centers.npy"

AGGREGATE_COLUMN = "aggregate_norm"

def target_norm_column(appliance):
    return f"{appliance}_norm"

def target_weight_column(appliance):
    return f"{appliance}_weight"

def load_building_dataframe(building):
    return pd.read_csv(csv_path(building))

def load_building_metadata(building):
    with open(metadata_path(building), "r", encoding="utf-8") as f:
        return json.load(f)

def get_appliance_metadata(building, appliance):
    meta = load_building_metadata(building)
    if appliance in meta and isinstance(meta[appliance], dict):
        return meta[appliance]
    return meta

def infer_columns(df, appliance):
    agg_col = AGGREGATE_COLUMN
    tgt_col = target_norm_column(appliance)
    if agg_col not in df.columns:
        raise ValueError(f"Missing aggregate column: {agg_col}")
    if tgt_col not in df.columns:
        raise ValueError(f"Missing target column: {tgt_col}")
    return agg_col, tgt_col

In [47]:
def inverse_target_transform_torch(y_norm, meta):
    mean_ = float(meta.get("target_mean", meta.get("mean", 0.0)))
    std_ = float(meta.get("target_std", meta.get("std", 1.0)))
    use_log = bool(meta.get("target_log1p", meta.get("use_log", False)))
    y_t = y_norm * std_ + mean_
    if use_log:
        y = torch.expm1(y_t)
    else:
        y = y_t
    return torch.clamp(y, min=0.0)

In [48]:
class BalancedClassifierDataset(Dataset):
    def __init__(self, aggregate, target, positive_centers, negative_centers, window_size=129, sample_weights=None):
        self.aggregate = aggregate.astype(np.float32)
        self.target = target.astype(np.float32)
        self.positive_centers = positive_centers.astype(np.int64)
        self.negative_centers = negative_centers.astype(np.int64)
        self.centers = np.concatenate([self.positive_centers, self.negative_centers]).astype(np.int64)
        self.labels = np.concatenate([
            np.ones(len(self.positive_centers), dtype=np.float32),
            np.zeros(len(self.negative_centers), dtype=np.float32)
        ])
        self.window_size = window_size
        self.half = window_size // 2
        self.sample_weights = np.ones(len(self.target), dtype=np.float32) if sample_weights is None else sample_weights.astype(np.float32)

    def __len__(self):
        return len(self.centers)

    def __getitem__(self, idx):
        c = int(self.centers[idx])
        l = c - self.half
        r = c + self.half + 1
        x = self.aggregate[l:r]
        y = self.target[c]
        w = self.sample_weights[c]
        label = self.labels[idx]
        x = torch.tensor(x, dtype=torch.float32).unsqueeze(-1)
        y = torch.tensor(y, dtype=torch.float32)
        label = torch.tensor(label, dtype=torch.float32)
        w = torch.tensor(w, dtype=torch.float32)
        return x, y, label, w

class ActiveOnlyRegressionDataset(Dataset):
    def __init__(self, aggregate, target, active_centers, window_size=129, sample_weights=None):
        self.aggregate = aggregate.astype(np.float32)
        self.target = target.astype(np.float32)
        self.centers = active_centers.astype(np.int64)
        self.window_size = window_size
        self.half = window_size // 2
        self.sample_weights = np.ones(len(self.target), dtype=np.float32) if sample_weights is None else sample_weights.astype(np.float32)

    def __len__(self):
        return len(self.centers)

    def __getitem__(self, idx):
        c = int(self.centers[idx])
        l = c - self.half
        r = c + self.half + 1
        x = self.aggregate[l:r]
        y = self.target[c]
        w = self.sample_weights[c]
        x = torch.tensor(x, dtype=torch.float32).unsqueeze(-1)
        y = torch.tensor(y, dtype=torch.float32)
        w = torch.tensor(w, dtype=torch.float32)
        return x, y, w

In [49]:
def make_sample_weights(target_array, power_weight=1.0):
    t = np.abs(target_array.astype(np.float32))
    return 1.0 + power_weight * t

def prepare_building_arrays(building, appliance):
    df = load_building_dataframe(building)
    agg_col, tgt_col = infer_columns(df, appliance)
    aggregate = df[agg_col].to_numpy(dtype=np.float32)
    target = df[tgt_col].to_numpy(dtype=np.float32)
    if target_weight_column(appliance) in df.columns:
        sample_weights = df[target_weight_column(appliance)].to_numpy(dtype=np.float32)
    else:
        sample_weights = make_sample_weights(target)
    balanced_centers = np.load(center_path(building, appliance, "balanced")).astype(np.int64)
    all_centers = np.load(center_path(building, appliance, "all")).astype(np.int64)
    inactive_centers = np.load(center_path(building, appliance, "inactive")).astype(np.int64)

    active_path = center_path(building, appliance, "active")
    if active_path.exists():
        active_centers = np.load(active_path).astype(np.int64)
    else:
        active_centers = np.array([c for c in all_centers if c not in set(inactive_centers.tolist())], dtype=np.int64)

    return {
        "aggregate": aggregate,
        "target": target,
        "weights": sample_weights,
        "balanced_centers": balanced_centers,
        "all_centers": all_centers,
        "inactive_centers": inactive_centers,
        "active_centers": active_centers
    }

In [50]:
def make_target_loaders(appliance, target_building, calibration_ratio=0.05, batch_size=256, window_size=129, mode="all"):
    data = prepare_building_arrays(target_building, appliance)
    aggregate = data["aggregate"]
    target = data["target"]
    weights = data["weights"]
    balanced = data["balanced_centers"].copy()
    all_centers = data["all_centers"].copy()
    inactive = data["inactive_centers"].copy()

    rng = np.random.default_rng(SEED)

    if mode == "all":
        pool = all_centers.copy()
        rng.shuffle(pool)
        n_cal = max(1, int(len(pool) * calibration_ratio))
        cal_centers = np.sort(pool[:n_cal])
    else:
        total_n = max(1, int(len(all_centers) * calibration_ratio))
        n_bal = int(total_n * 0.5)
        n_inactive = total_n - n_bal

        bal_pool = balanced.copy()
        inact_pool = inactive.copy()
        rng.shuffle(bal_pool)
        rng.shuffle(inact_pool)

        sel = []
        if len(bal_pool) > 0 and n_bal > 0:
            sel.append(bal_pool[:min(n_bal, len(bal_pool))])
        if len(inact_pool) > 0 and n_inactive > 0:
            sel.append(inact_pool[:min(n_inactive, len(inact_pool))])

        if len(sel) == 0:
            pool = all_centers.copy()
            rng.shuffle(pool)
            n_cal = max(1, int(len(pool) * calibration_ratio))
            cal_centers = np.sort(pool[:n_cal])
        else:
            cal_centers = np.unique(np.concatenate(sel)).astype(np.int64)
            cal_centers = np.sort(cal_centers)

    cal_set = set(cal_centers.tolist())
    full_holdout_centers = np.array([c for c in all_centers if int(c) not in cal_set], dtype=np.int64)
    if len(full_holdout_centers) == 0 and len(all_centers) > 0:
        full_holdout_centers = all_centers[:1]

    cal_ds = BalancedClassifierDataset(aggregate, target, cal_centers, np.array([], dtype=np.int64), window_size=window_size, sample_weights=weights)
    full_ds = BalancedClassifierDataset(aggregate, target, full_holdout_centers, np.array([], dtype=np.int64), window_size=window_size, sample_weights=weights)

    cal_loader = DataLoader(cal_ds, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    full_loader = DataLoader(full_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    return cal_loader, full_loader

def build_balanced_two_stage_loaders(appliance, target_building, calibration_ratio=0.05):
    cls_train_sets = []
    reg_train_sets = []

    for b in TRAIN_BUILDINGS:
        data = prepare_building_arrays(b, appliance)
        pos = data["active_centers"]
        neg = data["inactive_centers"]

        n = min(len(pos), len(neg))
        rng = np.random.default_rng(SEED)
        pos_sel = np.sort(rng.choice(pos, size=n, replace=False)) if len(pos) > n else pos.copy()
        neg_sel = np.sort(rng.choice(neg, size=n, replace=False)) if len(neg) > n else neg.copy()

        cls_ds = BalancedClassifierDataset(
            data["aggregate"], data["target"],
            pos_sel, neg_sel,
            window_size=WINDOW_SIZE,
            sample_weights=data["weights"]
        )
        cls_train_sets.append(cls_ds)

        reg_ds = ActiveOnlyRegressionDataset(
            data["aggregate"], data["target"],
            pos,
            window_size=WINDOW_SIZE,
            sample_weights=data["weights"]
        )
        reg_train_sets.append(reg_ds)

    cls_train_ds = ConcatDataset(cls_train_sets)
    reg_train_ds = ConcatDataset(reg_train_sets)

    target_data = prepare_building_arrays(target_building, appliance)
    tgt_pos = target_data["active_centers"]
    tgt_neg = target_data["inactive_centers"]

    n_val = min(len(tgt_pos), len(tgt_neg), 50000)
    rng = np.random.default_rng(SEED)
    if len(tgt_pos) > 0 and len(tgt_neg) > 0 and n_val > 0:
        val_pos = np.sort(rng.choice(tgt_pos, size=n_val, replace=False)) if len(tgt_pos) > n_val else tgt_pos.copy()
        val_neg = np.sort(rng.choice(tgt_neg, size=n_val, replace=False)) if len(tgt_neg) > n_val else tgt_neg.copy()
    else:
        val_pos = tgt_pos.copy()
        val_neg = tgt_neg.copy()

    cls_val_ds = BalancedClassifierDataset(
        target_data["aggregate"], target_data["target"],
        val_pos, val_neg,
        window_size=WINDOW_SIZE,
        sample_weights=target_data["weights"]
    )

    cls_train_loader = DataLoader(cls_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    reg_train_loader = DataLoader(reg_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    cls_val_loader = DataLoader(cls_val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    cal_loader, full_holdout_loader = make_target_loaders(
        appliance=appliance,
        target_building=target_building,
        calibration_ratio=calibration_ratio,
        batch_size=BATCH_SIZE,
        window_size=WINDOW_SIZE,
        mode=CALIBRATION_MODE
    )
    return cls_train_loader, reg_train_loader, cls_val_loader, cal_loader, full_holdout_loader

In [51]:
class ActivityClassifier(nn.Module):
    def __init__(self, input_dim=1, channels=64, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(input_dim, 32, 5, padding=2),
            nn.ReLU(),
            nn.Conv1d(32, channels, 5, padding=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(channels, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        return self.net(x).squeeze(-1)

class PowerRegressor(nn.Module):
    def __init__(self, input_dim=1, channels=64, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(input_dim, 32, 5, padding=2),
            nn.ReLU(),
            nn.Conv1d(32, channels, 5, padding=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(channels, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        return self.net(x).squeeze(-1)

class WeightedAsymmetricHuberLoss(nn.Module):
    def __init__(self, delta=1.0, underpredict_weight=2.0):
        super().__init__()
        self.delta = delta
        self.underpredict_weight = underpredict_weight

    def forward(self, pred, target, sample_weight=None):
        err = pred - target
        abs_err = torch.abs(err)
        huber = torch.where(abs_err <= self.delta, 0.5 * err ** 2, self.delta * (abs_err - 0.5 * self.delta))
        asym = torch.where(pred < target, self.underpredict_weight, 1.0)
        loss = huber * asym
        if sample_weight is not None:
            loss = loss * sample_weight
        return loss.mean()

class WeightedBCEWithLogitsLoss(nn.Module):
    def forward(self, logits, labels, sample_weight=None):
        loss = nn.functional.binary_cross_entropy_with_logits(logits, labels, reduction="none")
        if sample_weight is not None:
            loss = loss * sample_weight
        return loss.mean()

In [52]:
class WeightedAsymmetricHuberLoss(nn.Module):
    def __init__(self, delta=1.0, underpredict_weight=2.0):
        super().__init__()
        self.delta = delta
        self.underpredict_weight = underpredict_weight

    def forward(self, pred, target, sample_weight=None):
        err = pred - target
        abs_err = torch.abs(err)
        huber = torch.where(abs_err <= self.delta, 0.5 * err ** 2, self.delta * (abs_err - 0.5 * self.delta))
        asym = torch.where(pred < target, self.underpredict_weight, 1.0)
        loss = huber * asym
        if sample_weight is not None:
            loss = loss * sample_weight
        return loss.mean()

class WeightedBCEWithLogitsLoss(nn.Module):
    def forward(self, logits, labels, sample_weight=None):
        loss = nn.functional.binary_cross_entropy_with_logits(logits, labels, reduction="none")
        if sample_weight is not None:
            loss = loss * sample_weight
        return loss.mean()

def train_one_epoch_multitask(model, loader, optimizer, cls_loss_fn, reg_loss_fn, device, lambda_cls=1.0, lambda_reg=1.0):
    model.train()
    losses = []
    for x, y, label, w in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        label = label.to(device, non_blocking=True)
        w = w.to(device, non_blocking=True)

        optimizer.zero_grad()
        cls_logit, reg_out = model(x)

        cls_loss = cls_loss_fn(cls_logit, label, w)
        reg_loss = reg_loss_fn(reg_out, y, w)
        loss = lambda_cls * cls_loss + lambda_reg * reg_loss

        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    return float(np.mean(losses)) if len(losses) else 0.0

In [53]:
def train_one_epoch_classifier(model, loader, optimizer, loss_fn, device):
    model.train()
    losses = []
    for x, y, label, w in loader:
        x = x.to(device, non_blocking=True)
        label = label.to(device, non_blocking=True)
        w = w.to(device, non_blocking=True)
        optimizer.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, label, w)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return float(np.mean(losses)) if len(losses) else 0.0

def train_one_epoch_regressor(model, loader, optimizer, loss_fn, device):
    model.train()
    losses = []
    for batch in loader:
        if len(batch) == 4:
            x, y, label, w = batch
        elif len(batch) == 3:
            x, y, w = batch
        else:
            raise ValueError(f"Unexpected batch size: {len(batch)}")

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        w = w.to(device, non_blocking=True)

        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y, w)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    return float(np.mean(losses)) if len(losses) else 0.0

In [54]:
def collect_two_stage_outputs(classifier, regressor, loader, device, meta, prob_threshold=0.45):
    classifier.eval()
    regressor.eval()
    preds, trues, probs = [], [], []

    with torch.inference_mode():
        for batch in loader:
            if len(batch) == 4:
                x, y, label, w = batch
            elif len(batch) == 3:
                x, y, w = batch
            else:
                raise ValueError(f"Unexpected batch size: {len(batch)}")

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            prob = torch.sigmoid(classifier(x))
            reg = regressor(x)

            prob_np = prob.cpu().numpy()
            reg_np = inverse_target_transform_torch(reg, meta).cpu().numpy()
            true_np = inverse_target_transform_torch(y, meta).cpu().numpy()

            reg_np[prob_np < prob_threshold] = 0.0

            preds.append(reg_np)
            trues.append(true_np)
            probs.append(prob_np)

    if len(preds) == 0:
        return None, None, None

    return np.concatenate(preds), np.concatenate(trues), np.concatenate(probs)

def stage_metrics(y_true, y_pred):
    mae = float(np.mean(np.abs(y_true - y_pred)))
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = 0.0 if ss_tot == 0 else 1.0 - ss_res / ss_tot
    rel_mae_pct = float(100.0 * mae / max(np.mean(y_true), 1e-8))
    return {"mae": mae, "r2": r2, "rel_mae_pct": rel_mae_pct, "pred_mean": float(np.mean(y_pred)), "true_mean": float(np.mean(y_true))}

def binary_metrics(y_true, y_prob, thr=0.5, activity_threshold=10.0):
    true_on = (y_true >= activity_threshold).astype(np.int32)
    pred_on = (y_prob >= thr).astype(np.int32)
    tp = int(np.sum((true_on == 1) & (pred_on == 1)))
    tn = int(np.sum((true_on == 0) & (pred_on == 0)))
    fp = int(np.sum((true_on == 0) & (pred_on == 1)))
    fn = int(np.sum((true_on == 1) & (pred_on == 0)))
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    return {"tp": tp, "tn": tn, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1, "accuracy": acc}

In [55]:
def fit_linear_bias_correction(preds, trues):
    preds = np.asarray(preds, dtype=np.float64)
    trues = np.asarray(trues, dtype=np.float64)
    A = np.vstack([preds, np.ones_like(preds)]).T
    a, b = np.linalg.lstsq(A, trues, rcond=None)[0]
    return {"scale": float(a), "bias": float(b)}

def apply_linear_bias_correction(preds, correction):
    preds = np.asarray(preds, dtype=np.float64)
    out = preds * correction["scale"] + correction["bias"]
    return np.clip(out, 0.0, None)
def regression_metrics(y_true, y_pred):
    mae = float(np.mean(np.abs(y_true - y_pred)))
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = 0.0 if ss_tot == 0 else 1.0 - ss_res / ss_tot
    rel_mae_pct = float(100.0 * mae / max(np.mean(y_true), 1e-8))
    return {
        "mae": mae,
        "r2": r2,
        "rel_mae_pct": rel_mae_pct,
        "pred_mean": float(np.mean(y_pred)),
        "true_mean": float(np.mean(y_true)),
        "num_samples": len(y_true)
    }

def binary_metrics(y_true, y_prob, thr=0.45, activity_threshold=2.0):
    true_on = (y_true >= activity_threshold).astype(np.int32)
    pred_on = (y_prob >= thr).astype(np.int32)
    tp = int(np.sum((true_on == 1) & (pred_on == 1)))
    tn = int(np.sum((true_on == 0) & (pred_on == 0)))
    fp = int(np.sum((true_on == 0) & (pred_on == 1)))
    fn = int(np.sum((true_on == 1) & (pred_on == 0)))
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    return {"tp": tp, "tn": tn, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1, "accuracy": acc}

In [56]:
cls_train_loader, reg_train_loader, cls_val_loader, calibration_loader, full_holdout_loader = build_balanced_two_stage_loaders(
    APPLIANCE, TARGET_BUILDING, calibration_ratio=CALIBRATION_RATIO
)

source_meta = get_appliance_metadata(VAL_BUILDING, APPLIANCE)
target_meta = get_appliance_metadata(TARGET_BUILDING, APPLIANCE)

classifier = ActivityClassifier().to(DEVICE)
regressor = PowerRegressor().to(DEVICE)

cls_loss_fn = WeightedBCEWithLogitsLoss()
reg_loss_fn = WeightedAsymmetricHuberLoss(delta=1.0, underpredict_weight=2.0)

opt_cls = AdamW(classifier.parameters(), lr=CLASSIFIER_LR, weight_decay=1e-4)
opt_reg = AdamW(regressor.parameters(), lr=REGRESSOR_LR, weight_decay=1e-4)

for epoch in range(1, CLASSIFIER_EPOCHS + 1):
    t0 = time.time()
    cls_loss = train_one_epoch_classifier(classifier, cls_train_loader, opt_cls, cls_loss_fn, DEVICE)
    print(f"[CLS {epoch:02d}/{CLASSIFIER_EPOCHS}] loss={cls_loss:.4f} time={time.time()-t0:.1f}s")

for epoch in range(1, REGRESSOR_EPOCHS + 1):
    t0 = time.time()
    reg_loss = train_one_epoch_regressor(regressor, reg_train_loader, opt_reg, reg_loss_fn, DEVICE)
    print(f"[REG {epoch:02d}/{REGRESSOR_EPOCHS}] loss={reg_loss:.4f} time={time.time()-t0:.1f}s")

for epoch in range(1, CALIBRATION_EPOCHS + 1):
    t0 = time.time()
    reg_loss = train_one_epoch_regressor(regressor, calibration_loader, opt_reg, reg_loss_fn, DEVICE)
    print(f"[CAL {epoch:02d}/{CALIBRATION_EPOCHS}] loss={reg_loss:.4f} time={time.time()-t0:.1f}s")

[CLS 01/6] loss=1.4566 time=1.1s
[CLS 02/6] loss=1.0588 time=0.7s
[CLS 03/6] loss=0.7718 time=0.7s
[CLS 04/6] loss=0.5793 time=0.8s
[CLS 05/6] loss=0.4572 time=0.8s
[CLS 06/6] loss=0.3888 time=0.7s
[REG 01/6] loss=34.0034 time=0.3s
[REG 02/6] loss=32.3566 time=0.4s
[REG 03/6] loss=29.9542 time=0.4s
[REG 04/6] loss=25.9189 time=0.3s
[REG 05/6] loss=20.2991 time=0.4s
[REG 06/6] loss=17.6514 time=0.4s
[CAL 01/3] loss=0.7719 time=31.8s
[CAL 02/3] loss=0.3735 time=33.7s
[CAL 03/3] loss=0.3461 time=33.0s


In [57]:
preds, trues, probs = collect_two_stage_outputs(
    classifier, regressor, calibration_loader, DEVICE, target_meta, prob_threshold=PROB_THRESHOLD
)
correction = fit_linear_bias_correction(preds, trues)
print("Correction:", correction)

if preds is not None:
    reg = stage_metrics(trues, preds)
    binm = binary_metrics(trues, probs, thr=PROB_THRESHOLD, activity_threshold=ACTIVITY_THRESHOLD)

    print("REGRESSION")
    for k, v in reg.items():
        print(f"{k}: {v}")

    print("\nBINARY")
    for k, v in binm.items():
        print(f"{k}: {v}")

Correction: {'scale': 0.45503085242725355, 'bias': 0.0482244445016486}
REGRESSION
mae: 0.27581527829170227
r2: -0.20764347177557574
rel_mae_pct: 115.55645751953125
pred_mean: 0.41856497526168823
true_mean: 0.23868443071842194

BINARY
tp: 40814
tn: 726335
fp: 41726
fn: 8344
precision: 0.4944754058638236
recall: 0.8302616054355344
f1: 0.6198119941077314
accuracy: 0.9387312336105744


In [58]:
full_preds, full_trues, full_probs = collect_two_stage_outputs(
    classifier, regressor, full_holdout_loader, DEVICE, target_meta, prob_threshold=PROB_THRESHOLD
)

corrected_preds = apply_linear_bias_correction(full_preds, correction)

reg = regression_metrics(full_trues, corrected_preds)
binm = binary_metrics(full_trues, full_probs, thr=PROB_THRESHOLD, activity_threshold=ACTIVITY_THRESHOLD)

print("CORRECTED REGRESSION")
for k, v in reg.items():
    print(f"{k}: {v}")

print("\nBINARY")
for k, v in binm.items():
    print(f"{k}: {v}")

CORRECTED REGRESSION
mae: 0.2748460106996593
r2: 0.3927494658937256
rel_mae_pct: 115.68286895751953
pred_mean: 0.2375844017519467
true_mean: 0.23758575320243835
num_samples: 15527166

BINARY
tp: 770303
tn: 13808884
fp: 789492
fn: 158487
precision: 0.49384887116576215
recall: 0.8293618579011401
f1: 0.6190690693707468
accuracy: 0.9389470686408582


In [59]:
def sweep_prob_thresholds(classifier, regressor, loader, device, meta,
                          thresholds=(0.3, 0.4, 0.5, 0.6, 0.7, 0.8),
                          activity_threshold=2.0):
    classifier.eval()
    regressor.eval()

    all_probs = []
    all_preds = []
    all_trues = []

    with torch.inference_mode():
        for x, y, label, w in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            prob = torch.sigmoid(classifier(x))
            reg = regressor(x)

            prob_np = prob.cpu().numpy()
            reg_np = inverse_target_transform_torch(reg, meta).cpu().numpy()
            true_np = inverse_target_transform_torch(y, meta).cpu().numpy()

            all_probs.append(prob_np)
            all_preds.append(reg_np)
            all_trues.append(true_np)

    if len(all_probs) == 0:
        return pd.DataFrame(columns=["threshold", "mae", "r2", "rel_mae_pct", "pred_mean", "true_mean", "precision", "recall", "f1", "accuracy"])

    probs = np.concatenate(all_probs)
    preds_raw = np.concatenate(all_preds)
    trues = np.concatenate(all_trues)

    rows = []
    for thr in thresholds:
        preds = preds_raw.copy()
        preds[probs < thr] = 0.0

        reg = stage_metrics(trues, preds)
        binm = binary_metrics(trues, probs, thr=thr, activity_threshold=activity_threshold)

        rows.append({
            "threshold": thr,
            "mae": reg["mae"],
            "r2": reg["r2"],
            "rel_mae_pct": reg["rel_mae_pct"],
            "pred_mean": reg["pred_mean"],
            "true_mean": reg["true_mean"],
            "precision": binm["precision"],
            "recall": binm["recall"],
            "f1": binm["f1"],
            "accuracy": binm["accuracy"]
        })

    return pd.DataFrame(rows)

In [60]:
sweep_df = sweep_prob_thresholds(
    classifier=classifier,
    regressor=regressor,
    loader=full_holdout_loader,
    device=DEVICE,
    meta=target_meta,
    thresholds=(0.3, 0.4, 0.5, 0.6, 0.7, 0.8),
    activity_threshold=ACTIVITY_THRESHOLD
)

print(sweep_df)

   threshold       mae        r2  rel_mae_pct  pred_mean  true_mean  \
0        0.3  0.273729 -0.188265   115.212677   0.447036   0.237586   
1        0.4  0.272857 -0.193376   114.845551   0.433151   0.237586   
2        0.5  0.274484 -0.205244   115.530518   0.416147   0.237586   
3        0.6  0.279369 -0.228393   117.586571   0.395838   0.237586   
4        0.7  0.284630 -0.253918   119.800964   0.369600   0.237586   
5        0.8  0.294843 -0.299339   124.099403   0.342367   0.237586   

   precision    recall        f1  accuracy  
0   0.492091  0.923708  0.642109  0.938407  
1   0.495693  0.883285  0.635018  0.939265  
2   0.493849  0.829362  0.619069  0.938947  
3   0.484725  0.759331  0.591721  0.937320  
4   0.472317  0.673953  0.555401  0.935457  
5   0.446627  0.574292  0.502477  0.931972  


In [61]:
# preds, trues, probs = collect_two_stage_outputs(
#     classifier, regressor, full_holdout_loader, DEVICE, target_meta, prob_threshold=PROB_THRESHOLD
# )

# if preds is not None:
#     reg = stage_metrics(trues, preds)
#     binm = binary_metrics(trues, probs, thr=PROB_THRESHOLD, activity_threshold=ACTIVITY_THRESHOLD)

#     print("REGRESSION")
#     for k, v in reg.items():
#         print(f"{k}: {v}")

#     print("\nBINARY")
#     for k, v in binm.items():
#         print(f"{k}: {v}")

In [62]:
best_threshold = 0.45

preds, trues, probs = collect_two_stage_outputs(
    classifier, regressor, full_holdout_loader, DEVICE, target_meta, prob_threshold=best_threshold
)

reg = stage_metrics(trues, preds)
binm = binary_metrics(trues, probs, thr=best_threshold, activity_threshold=ACTIVITY_THRESHOLD)

print("REGRESSION")
for k, v in reg.items():
    print(f"{k}: {v}")

print("\nBINARY")
for k, v in binm.items():
    print(f"{k}: {v}")

REGRESSION
mae: 0.27301061153411865
r2: -0.1971759321695994
rel_mae_pct: 114.91034698486328
pred_mean: 0.42492982745170593
true_mean: 0.23758575320243835

BINARY
tp: 797492
tn: 13788084
fp: 810292
fn: 131298
precision: 0.49601936578545375
recall: 0.8586354288913532
f1: 0.6287945867142058
accuracy: 0.9393585410241637
